# Build Python-prepared dashboard data

This notebook reads `data/seoul.geojson`, prepares a minimal `chart_data` dictionary, and saves it as `data/chart_data.json` so the compile step can embed it into the dashboard.

## Preview the first trip parquet file

Use the next cells to inspect the first available trip parquet file before designing the OD flow aggregation pipeline.

In [1]:
from pathlib import Path
import json
from statistics import mean

project_dir = Path.cwd()
trip_dir = project_dir.parent / "static-presentation" / "input" / "trip_parquet"
station_csv_path = project_dir.parent / "static-presentation" / "input" / "Station.csv"
geojson_path = project_dir / "data" / "seoul.geojson"
output_path = project_dir / "data" / "chart_data.json"

trip_dir, station_csv_path, geojson_path, output_path

(PosixPath('/Users/yunsik/Library/CloudStorage/OneDrive-Personal/Charlotte/Coursework/04-Spring2026/GEOG8005/GeoViz_Live/Project/static-presentation/input/trip_parquet'),
 PosixPath('/Users/yunsik/Library/CloudStorage/OneDrive-Personal/Charlotte/Coursework/04-Spring2026/GEOG8005/GeoViz_Live/Project/static-presentation/input/Station.csv'),
 PosixPath('/Users/yunsik/Library/CloudStorage/OneDrive-Personal/Charlotte/Coursework/04-Spring2026/GEOG8005/GeoViz_Live/Project/interactive/data/seoul.geojson'),
 PosixPath('/Users/yunsik/Library/CloudStorage/OneDrive-Personal/Charlotte/Coursework/04-Spring2026/GEOG8005/GeoViz_Live/Project/interactive/data/chart_data.json'))

In [2]:
# Find the first daily parquet file in sorted order.
# This should point to `trip_2025-01-01.parquet` if all files are present.
trip_files = sorted(trip_dir.glob("trip_*.parquet"))
first_trip_path = trip_files[0]

print(f"First trip parquet file: {first_trip_path.name}")
first_trip_path

First trip parquet file: trip_2025-01-01.parquet


PosixPath('/Users/yunsik/Library/CloudStorage/OneDrive-Personal/Charlotte/Coursework/04-Spring2026/GEOG8005/GeoViz_Live/Project/static-presentation/input/trip_parquet/trip_2025-01-01.parquet')

In [3]:
# Read the first parquet file with pandas.
# Note:
# - This cell expects `pandas` and a parquet engine such as `pyarrow` to be installed.
# - If the imports fail, install them in your notebook environment first.
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError(
        "pandas is required to inspect the trip parquet file. "
        "Install pandas and pyarrow in the notebook environment, then rerun this cell."
    ) from exc

trip_df = pd.read_parquet(first_trip_path)

print(f"Row count: {len(trip_df):,}")
print(f"Column count: {len(trip_df.columns)}")
trip_df.head(3)

Row count: 49,991
Column count: 6


,RENT_DT,RENT_STATION_ID,RTN_DT,RETURN_STATION_ID,USE_MIN,USE_DST
0,2025-01-01 00:01:39,ST-1678,2025-01-01 00:03:08,ST-2375,1,290.0
1,2025-01-01 00:00:10,ST-3251,2025-01-01 00:03:15,ST-2044,3,450.0
2,2025-01-01 00:02:12,ST-2822,2025-01-01 00:03:50,ST-1649,1,0.0


In [4]:
# Print a compact schema summary so you can identify join keys,
# timestamps, station identifiers, and potential OD attributes.
schema_df = pd.DataFrame({
    "column": trip_df.columns,
    "dtype": trip_df.dtypes.astype(str).values,
    "non_null_count": trip_df.notna().sum().values,
    "sample_value": [trip_df[col].dropna().iloc[0] if trip_df[col].notna().any() else None for col in trip_df.columns]
})

schema_df

,column,dtype,non_null_count,sample_value
0,RENT_DT,datetime64[us],49991,2025-01-01 00:01:39
1,RENT_STATION_ID,str,49991,ST-1678
2,RTN_DT,datetime64[us],49991,2025-01-01 00:03:08
3,RETURN_STATION_ID,str,49825,ST-2375
4,USE_MIN,int64,49991,1
5,USE_DST,float64,49991,290.0


In [5]:
# Show the first 10 rows to inspect the raw trip record structure.
trip_df.head(10)

,RENT_DT,RENT_STATION_ID,RTN_DT,RETURN_STATION_ID,USE_MIN,USE_DST
0,2025-01-01 00:01:39,ST-1678,2025-01-01 00:03:08,ST-2375,1,290.00
1,2025-01-01 00:00:10,ST-3251,2025-01-01 00:03:15,ST-2044,3,450.00
2,2025-01-01 00:02:12,ST-2822,2025-01-01 00:03:50,ST-1649,1,0.00
3,2025-01-01 00:01:36,ST-919,2025-01-01 00:04:00,ST-3194,2,1230.00
4,2025-01-01 00:00:41,ST-3143,2025-01-01 00:04:11,ST-1757,3,567.92
5,2025-01-01 00:00:39,ST-1059,2025-01-01 00:05:05,ST-2745,4,787.39
6,2025-01-01 00:01:09,ST-3286,2025-01-01 00:05:22,ST-3258,4,338.31
7,2025-01-01 00:00:10,ST-72,2025-01-01 00:05:40,ST-307,5,557.66
8,2025-01-01 00:04:38,ST-3316,2025-01-01 00:05:49,ST-3316,1,0.00
9,2025-01-01 00:01:28,ST-2601,2025-01-01 00:05:51,ST-1409,4,470.00


In [ ]:
# Read the current Seoul boundary GeoJSON.
geojson_data = json.loads(geojson_path.read_text(encoding="utf-8"))
features = geojson_data.get("features", [])
properties = features[0].get("properties", {}) if features else {}

properties

In [ ]:
# Flatten nested coordinates so we can estimate a center point without
# requiring extra geospatial libraries in this notebook.
points = []

def walk_coordinates(node):
    if isinstance(node, list) and node:
        if isinstance(node[0], (int, float)) and len(node) >= 2:
            points.append((node[0], node[1]))
        else:
            for child in node:
                walk_coordinates(child)

for feature in features:
    geometry = feature.get("geometry", {})
    walk_coordinates(geometry.get("coordinates", []))

center_lon = mean([point[0] for point in points])
center_lat = mean([point[1] for point in points])
center_lat, center_lon

In [ ]:
# Build a browser-friendly data object.
# Later you can expand this with station-level, hourly, or chart-specific arrays.
base_year = int(str(properties.get("BASE_DATE", "2025"))[:4])

chart_data = {
    "meta": {
        "pageTitle": "Seoul Bike Dashboard",
        "dashboardTitle": f"Seoul Public Bike Interactive Dashboard ({base_year})",
        "dashboardSubtitle": "Template wired for Python-processed data through window.CHART_DATA",
        "sourceNote": "This dashboard is compiled from Python-prepared JSON data and consumed by script.js in the browser.",
        "lastUpdated": properties.get("BASE_DATE", "")
    },
    "filters": {
        "years": [base_year],
        "defaultYear": base_year,
        "hours": list(range(24)),
        "defaultHour": 12
    },
    "map": {
        "center": [center_lat, center_lon],
        "zoom": 11,
        "boundaryGeoJsonPath": "data/seoul.geojson"
    },
    "chart": {
        "bars": [
            {
                "label": properties.get("SIDO_NM", "Seoul"),
                "value": len(features)
            }
        ]
    },
    "stationMarkers": [
        {
            "id": "seoul-centroid",
            "name": "Seoul Example Record",
            "lat": center_lat,
            "lon": center_lon,
            "year": base_year,
            "hour": 12,
            "rentals": 1
        }
    ],
    "summary": {
        "totalRegions": len(features),
        "totalRecords": 1
    }
}

chart_data

In [ ]:
# Save the processed data so the compile notebook can inject it into HTML.
output_path.write_text(
    json.dumps(chart_data, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"Saved processed dashboard data to: {output_path}")

In [14]:
import geopandas as gpd

seoul = gpd.read_file('data/seoul_dong.shp')
seoul = seoul.to_crs(epsg=4326)
seoul.head()

,BASE_DATE,ADM_CD,ADM_NM,gu_code,geometry
0,20250630,11010530,Sajik,1101,"POLYGON ((126.97399 37.57823, 126.974 37.57809..."
1,20250630,11010540,Samcheong,1101,"POLYGON ((126.97714 37.59768, 126.9773 37.5976..."
2,20250630,11010550,Buam,1101,"POLYGON ((126.96173 37.60714, 126.96182 37.607..."
3,20250630,11010560,Pyeongchang,1101,"POLYGON ((126.97508 37.63118, 126.97488 37.630..."
4,20250630,11010570,Muak,1101,"POLYGON ((126.95975 37.58001, 126.96006 37.579..."


In [19]:
seoul_dissolved = seoul.dissolve(
    by="gu_code",
    aggfunc={
        "ADM_NM": "first",
        "BASE_DATE": "first"
    }
).reset_index()

seoul_dissolved.head(25)

,gu_code,geometry,ADM_NM,BASE_DATE
0,1101,"POLYGON ((126.96883 37.56798, 126.96866 37.567...",Sajik,20250630
1,1102,"POLYGON ((127.00421 37.55015, 127.00419 37.550...",Sogong,20250630
2,1103,"POLYGON ((126.9824 37.50653, 126.981 37.50653,...",Huam,20250630
3,1104,"POLYGON ((127.03479 37.53583, 127.03473 37.535...",Wangsimni2,20250630
4,1105,"POLYGON ((127.07205 37.52342, 127.07035 37.523...",Hwayang,20250630
5,1106,"POLYGON ((127.05495 37.56516, 127.05485 37.565...",Hoegi,20250630
6,1107,"POLYGON ((127.09375 37.57057, 127.09337 37.570...",Myeonmok2,20250630
7,1108,"POLYGON ((127.02427 37.57909, 127.02425 37.579...",Donam1,20250630
8,1109,"POLYGON ((127.02188 37.61233, 127.02181 37.612...",Beon1,20250630
9,1110,"POLYGON ((127.03642 37.63686, 127.03607 37.636...",Ssangmun1,20250630


In [21]:
ADM_map = {
    'Sajik': 'Jongno', 'Sogong': 'Jung', 'Huam': 'Yongsan', 'Wangsimni2': 'Seongdong', 'Hwayang': 'Gwangjin', 'Hoegi': 'Dongdaemun',
    'Myeonmok2': 'Jungrang', 'Donam1': 'Seongbuk', 'Beon1': 'Gangbuk', 'Ssangmun1': 'Dobong', 'Wolgye1': 'Nowon', 'Nokbeon': 'Eunpyeong',
    'Cheonyeon': 'Seodaemun', 'Yonggang': 'Mapo', 'Mok1': 'Yangcheon', 'Yeomchang': 'Gangseo', 'Sindorim': 'Guro', 'Gasan': 'Geumcheon',
    'Yeoui': 'Yeongdeungpo', 'Noryangjin2': 'Dongjak', 'Boramae': 'Gwanak', 'Seocho1': 'Seocho', 'Sinsa': 'Gangnam', 'Pungnap1': 'Songpa', 'Myeongil1': 'Gangdong'
}

In [22]:
seoul_dissolved["ADM_NM"] = seoul_dissolved["ADM_NM"].map(ADM_map).fillna(seoul_dissolved["ADM_NM"])
seoul_dissolved.head()

,gu_code,geometry,ADM_NM,BASE_DATE
0,1101,"POLYGON ((126.96883 37.56798, 126.96866 37.567...",Jongno,20250630
1,1102,"POLYGON ((127.00421 37.55015, 127.00419 37.550...",Jung,20250630
2,1103,"POLYGON ((126.9824 37.50653, 126.981 37.50653,...",Yongsan,20250630
3,1104,"POLYGON ((127.03479 37.53583, 127.03473 37.535...",Seongdong,20250630
4,1105,"POLYGON ((127.07205 37.52342, 127.07035 37.523...",Gwangjin,20250630


In [23]:
seoul_dissolved.to_file('data/seoul_gu.geojson', index=False)